In [12]:
import re
import json
import numpy as np
import pandas as pd
from collections import Counter
from monty.serialization import loadfn

In [13]:
df_entry = {
    "id": [],
    "representation": [],
    "energy_above_hull": [],
    "unit_cell": [],
    "band_gap": []
}

for i in range(1, 5):
    data = loadfn(f"/home/qfu1/data/CleanData/clean_data{i}.json.gz")
    df_entry["id"].extend(data.get("id"))
    df_entry["representation"].extend(data.get("representation"))
    df_entry["energy_above_hull"].extend(data.get("energy_above_hull"))
    df_entry["unit_cell"].extend(data.get("unit_cell"))
    df_entry["band_gap"].extend(data.get("band_gap"))
    
    
MPdata = pd.DataFrame(df_entry).dropna()
MPdata

,id,representation,energy_above_hull,unit_cell,band_gap
0,mp-1094120,P 1 1 1 Nb 1a 0.069 0.14 0.503 Nb 1a 0.51 0.21...,0.189748,"{'a': 5.374343, 'b': 6.938010689619107, 'c': 7...",0.0000
1,mp-1120447,P 1 1 1 Si 1a 0.236 0.023 0.117 Si 1a 0.556 0....,0.372510,"{'a': 4.99306507357568, 'b': 5.386323637701111...",0.0000
2,mp-1179802,P 1 1 1 Rb 1a 0.389 0.795 0.234 Rb 1a 0.975 0....,0.053649,"{'a': 8.978624, 'b': 9.419573209459385, 'c': 9...",0.0000
3,mp-1180008,P 1 1 1 O 1a 0.422 0.73 0.499 O 1a 0.59 0.281 ...,0.419854,"{'a': 4.930603, 'b': 5.122182314209152, 'c': 7...",1.2773
4,mp-1180064,P 1 1 1 O 1a 0.866 0.5 0.989 O 1a 0.095 0.426 ...,0.387014,"{'a': 5.160296, 'b': 5.2402447696794665, 'c': ...",0.0423
...,...,...,...,...,...
19486,mp-2499,C m c m Ba 4c 0.000 0.14 0.000 Si 4c 0.000 0.4...,0.000000,"{'a': 5.0742047285416625, 'b': 12.125411970413...",0.0000
19487,mp-25233,C m c m V 8f 0.000 0.296 0.604 O 4c 0.000 0.76...,0.032813,"{'a': 3.537850599999998, 'b': 10.04929022, 'c'...",2.4212
19488,mp-255,C m c m Na 4c 0.000 0.178 0.000 Na 4c 0.000 0....,0.000000,"{'a': 7.1306631509984335, 'b': 10.760536057615...",0.0000
19489,mp-2580,C m c m Nb 4c 0.000 0.145 0.000 B 4c 0.000 0.4...,0.000000,"{'a': 3.312388129461378, 'b': 8.78111441729912...",0.0000


In [14]:
with open('RandomGenDATA.json', 'r') as f:
    RandomGenDATA = json.load(f)

In [15]:
REPRESENTATION = [RandomGenDATA[f'{idx}']['REPRESENTATION'] for idx in range(len(RandomGenDATA.keys()))]
energy_above_hull = [float(0)]*len(REPRESENTATION)
BANDGAP = [float(0)]*len(REPRESENTATION)
ID = [np.nan]*len(REPRESENTATION)

REPRESENTATION.extend(MPdata['representation'])
energy_above_hull.extend(MPdata['energy_above_hull'])
BANDGAP.extend(MPdata['band_gap'])
ID.extend(MPdata['id'])

In [16]:
df = pd.DataFrame({"id": ID, "string": REPRESENTATION, "band_gap": BANDGAP, "energy_above_hull": energy_above_hull})

In [17]:
df.head()

,id,string,band_gap,energy_above_hull
0,NaN,P 4bar n 2 Tb 4f 0.355 0.000 0.000,0.0,0.0
1,NaN,P 6bar 1 1 Eu 2g 0.000 0.000 0.545 C 2g 0.000 ...,0.0,0.0
2,NaN,I 4bar c 2 Tl 4a 0.000 0.000 0.000 Cl 8f 0.000...,0.0,0.0
3,NaN,P m c 21 Kr 4c 0.595 0.217 0.468 Tb 2a 0.000 0...,0.0,0.0
4,NaN,P 3bar 1 c Co 2c 0.000 0.000 0.000,0.0,0.0


In [18]:
def process_string(s):
    tokens = s.split()
    tokens.insert(0, "START")
    tokens.append("END")
    return tokens

In [19]:
def split_float_values(token_list):
    new_tokens = []
    for token in token_list:
        if re.match(r'^\d+\.\d+$', token):
            new_tokens.extend(list(token.replace('0.', '')))
        else:
            new_tokens.append(token)
    return new_tokens

In [20]:
df['tokens'] = df['string'].apply(process_string)
df['tokens'] = df['tokens'].apply(split_float_values)
df['tokens'].head()

0    [START, P, 4bar, n, 2, Tb, 4f, 3, 5, 5, 0, 0, ...
1    [START, P, 6bar, 1, 1, Eu, 2g, 0, 0, 0, 0, 0, ...
2    [START, I, 4bar, c, 2, Tl, 4a, 0, 0, 0, 0, 0, ...
3    [START, P, m, c, 21, Kr, 4c, 5, 9, 5, 2, 1, 7,...
4    [START, P, 3bar, 1, c, Co, 2c, 0, 0, 0, 0, 0, ...
Name: tokens, dtype: object

In [21]:
def VocabDict(data):
    segment1_list = []
    segment2_list = []
    
    for tokens in data:
        segment1 = tokens[1:5] # spacegroup
        segment2 = tokens[5:] # wyckoff info
        segment2.insert(0, tokens[0])
        
        segment1_list.extend([f's1_{token}' for token in segment1])
        segment2_list.extend([f's2_{token}' for token in segment2])
    
    counter1 = Counter(segment1_list)
    counter2 = Counter(segment2_list)
    combined_counter = counter1 + counter2
    combined_items = combined_counter.most_common()
    token_to_number_combined = {token: i+1 for i, (token, _) in enumerate(combined_items)}
    number_to_token_combined = dict(zip(token_to_number_combined.values(), token_to_number_combined.keys()))
    return token_to_number_combined, number_to_token_combined

In [22]:
token_to_number_combined, number_to_token_combined = VocabDict(df['tokens'])

In [23]:
def tokenize_string(tokens):
    segment1 = tokens[1:5]
    segment2 = tokens[5:]
    
    tokenized_segment1 = [token_to_number_combined["s1_"+token] for token in segment1]
    tokenized_segment2 = [token_to_number_combined["s2_"+token] for token in segment2]
    
    return [token_to_number_combined["s2_"+tokens[0]]] + tokenized_segment1 + tokenized_segment2

In [24]:
df['tokenized'] = df['tokens'].apply(tokenize_string)

In [25]:
df

,id,string,band_gap,energy_above_hull,tokens,tokenized
0,NaN,P 4bar n 2 Tb 4f 0.355 0.000 0.000,0.0000,0.000000,"[START, P, 4bar, n, 2, Tb, 4f, 3, 5, 5, 0, 0, ...","[13, 15, 101, 81, 47, 107, 58, 2, 8, 8, 1, 1, ..."
1,NaN,P 6bar 1 1 Eu 2g 0.000 0.000 0.545 C 2g 0.000 ...,0.0000,0.000000,"[START, P, 6bar, 1, 1, Eu, 2g, 0, 0, 0, 0, 0, ...","[13, 15, 89, 17, 17, 145, 152, 1, 1, 1, 1, 1, ..."
2,NaN,I 4bar c 2 Tl 4a 0.000 0.000 0.000 Cl 8f 0.000...,0.0000,0.000000,"[START, I, 4bar, c, 2, Tl, 4a, 0, 0, 0, 0, 0, ...","[13, 34, 101, 24, 47, 122, 27, 1, 1, 1, 1, 1, ..."
3,NaN,P m c 21 Kr 4c 0.595 0.217 0.468 Tb 2a 0.000 0...,0.0000,0.000000,"[START, P, m, c, 21, Kr, 4c, 5, 9, 5, 2, 1, 7,...","[13, 15, 11, 24, 104, 277, 33, 8, 10, 8, 6, 3,..."
4,NaN,P 3bar 1 c Co 2c 0.000 0.000 0.000,0.0000,0.000000,"[START, P, 3bar, 1, c, Co, 2c, 0, 0, 0, 0, 0, ...","[13, 15, 20, 17, 24, 72, 52, 1, 1, 1, 1, 1, 1,..."
...,...,...,...,...,...,...
19084,mp-2499,C m c m Ba 4c 0.000 0.14 0.000 Si 4c 0.000 0.4...,0.0000,0.000000,"[START, C, m, c, m, Ba, 4c, 0, 0, 0, 1, 4, 0, ...","[13, 40, 11, 24, 11, 129, 33, 1, 1, 1, 3, 9, 1..."
19085,mp-25233,C m c m V 8f 0.000 0.296 0.604 O 4c 0.000 0.76...,2.4212,0.032813,"[START, C, m, c, m, V, 8f, 0, 0, 0, 2, 9, 6, 6...","[13, 40, 11, 24, 11, 71, 38, 1, 1, 1, 6, 10, 4..."
19086,mp-255,C m c m Na 4c 0.000 0.178 0.000 Na 4c 0.000 0....,0.0000,0.000000,"[START, C, m, c, m, Na, 4c, 0, 0, 0, 1, 7, 8, ...","[13, 40, 11, 24, 11, 106, 33, 1, 1, 1, 3, 5, 7..."
19087,mp-2580,C m c m Nb 4c 0.000 0.145 0.000 B 4c 0.000 0.4...,0.0000,0.000000,"[START, C, m, c, m, Nb, 4c, 0, 0, 0, 1, 4, 5, ...","[13, 40, 11, 24, 11, 85, 33, 1, 1, 1, 3, 9, 8,..."


In [26]:
df.to_json("CleanData(Tokenized).json", orient="records", indent=4)

In [27]:
with open("token_to_number_combined.json", "w") as f:
    json.dump(token_to_number_combined, f, indent=4)

with open("number_to_token_combined.json", "w") as f:
    json.dump(number_to_token_combined, f, indent=4)

In [28]:
df = pd.read_json("CleanData(Tokenized).json", orient="records")
df.head()

,id,string,band_gap,energy_above_hull,tokens,tokenized
0,None,P 4bar n 2 Tb 4f 0.355 0.000 0.000,0.0,0.0,"[START, P, 4bar, n, 2, Tb, 4f, 3, 5, 5, 0, 0, ...","[13, 15, 101, 81, 47, 107, 58, 2, 8, 8, 1, 1, ..."
1,None,P 6bar 1 1 Eu 2g 0.000 0.000 0.545 C 2g 0.000 ...,0.0,0.0,"[START, P, 6bar, 1, 1, Eu, 2g, 0, 0, 0, 0, 0, ...","[13, 15, 89, 17, 17, 145, 152, 1, 1, 1, 1, 1, ..."
2,None,I 4bar c 2 Tl 4a 0.000 0.000 0.000 Cl 8f 0.000...,0.0,0.0,"[START, I, 4bar, c, 2, Tl, 4a, 0, 0, 0, 0, 0, ...","[13, 34, 101, 24, 47, 122, 27, 1, 1, 1, 1, 1, ..."
3,None,P m c 21 Kr 4c 0.595 0.217 0.468 Tb 2a 0.000 0...,0.0,0.0,"[START, P, m, c, 21, Kr, 4c, 5, 9, 5, 2, 1, 7,...","[13, 15, 11, 24, 104, 277, 33, 8, 10, 8, 6, 3,..."
4,None,P 3bar 1 c Co 2c 0.000 0.000 0.000,0.0,0.0,"[START, P, 3bar, 1, c, Co, 2c, 0, 0, 0, 0, 0, ...","[13, 15, 20, 17, 24, 72, 52, 1, 1, 1, 1, 1, 1,..."
